# (리뷰) MPC, Lazic2018

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [제어]

### About this doc 

`-` 논문리뷰

`-` 학부수준 

`-` 이 포스트는 아래논문의 리뷰임. 

Lazic, N., Boutilier, C., Lu, T., Wong, E., Roy, B., Ryu, M. K., & Imwalle, G. (2018). Data center cooling using model-predictive control. In Advances in Neural Information Processing Systems (pp. 3818-3827).



### Model Predictive control 

`-` 저자들은 PID에 존재하는 불필요한 성분들을 제거하기 위해서 **MPC**를 도입한다고 주장한다. 여기에서 **MPC**는 model predictive control의 약자이다.

`-` 저자들이 멀하고 싶은지 알기 위하여 잠시 배경지식을 공부하여 보자. 저자들이 하고 싶은건 서버실의 온도제어이다. 서버실의 온도를 제어하는 방법은 대충 다음과 같은것 같다. **(1)** 서버실밖의 쿨링타워에서는 차가운 물을 서버실로 보낸다. 이때 찬물의 온도를 **EWT** 라고 한다. **(2)** 차가운 물이 관을 타고 가서 서버실에 있는 에어컨에 도착한다. **(3)** 에어컨 에서는 서버실의 (뜨거운) 공기를 흡수한다. 그리고 (2)의 과정에 의해 도착한 차가운 물을 이용해 흡수된 (뜨거운) 공기를 찬공기로 만들어 배출한다. 이때 에어컨으로 흡수되는 뜨거운 공기의 온도를 **EAT** 라고 한다. 그리고 에어컨이 내뿜는 차가운 공기의 온도를 **LAT** 라고 한다. **(4-1)** 이때 에어컨이 찬공기를 그냥 보내는 것이 아니다. 서버실 전체에 골고루 찬공기가 갈 수 있게 적절한 바람세기로 보내야 한다. 그래서 서버실의 공기압력을 체크하여 어느정도의 강도로 팬을 돌릴지 결정한다. 공기압력은 여기에서 **DP** 로 정의하고 팬의강도는 **fan speed** 라고 정의한다. **(4-2)** 또한 얼마만큼 냉각을 시켜 공기를 보내야 할지 결정해야 하므로 서버실 온도 **CAT** 와 에어컨으로 유입되는 공기의 온도 **EAT** 를 보고 벨브를 얼마나 열지 결정한다. 이때 벨브를 얼마나 여는지를 측정하는 변수를 **valve opening** 라고 정의한다. **(5)** 에어컨에 차가운물이 들어가서 찬공기를 만들었으므로 에어컨에서 배출된 물은 들어온물보다 따듯한 물일 것이다. 이때의 물의 온도를 **LWT** 라고 한다. 이 물은 다시 관을 타고 서버실 밖의 쿨링타워로 빠져나간다. **(6)** 쿨링타워로 들어온 물은 다시 냉각되어서 서버실로 보내진다. 

`-` 위에서 정의된 각 약어의 의미는 아래와 같다. 
  - **EWT**: entering water temperature
  - **EAT**: entering air temperature 
  - **LAT**: leaving air temperature 
  - **LWT**: leaving water temperature 
  - **DP**: differential air pressure
  - **CAT**: cold-aisle temperature

`-` 지금까지 언급된 변수들은 크게 3가지 카테고리에 들어갈 수 있다. 첫번째는 **상태변수** 이다. 상태변수는 우리가 예측하고 싶거나 조절하고 싶은 변수이다. 예를들면 서버실의 현재 온도 **CAT**, 에어컨으로 들어오는 공기의 온도 **EAT**, 에어컨에서 빠져나가는 공기 온도 **LAT**, 그리고 마지막으로 서버실의 압력상태(이걸 알아야 얼만큼 강하게 팬을 돌려야 할지 결정할 수 있음)를 의미하는 **DP** 가 있다. 두번째 변수는 **제어변수** 이다. 제어변수는 상태변수에 따라서 우리가 구체적으로 어떠한 액션을 해야할지를 결정하는 변수이다. 제어변수에 해당하는것은 에어컨에서 찬물의 밸브를 얼마나 열지 결정하는 **valve opening**, 그리고 에어컨에서 나오는 바람의 세기를 결정하는 **fan speed**가 있다. 마지막 변수는 **장애변수** 이다. 장애변수의 의미는 우리가 컨트롤하지 못하는 어떠한 외부요인에 의해 휘둘리는 변수라고 생각하면 된다. 장애변수의 예는 서버실로 들어가는 물의 온도인 **EWT** 와 **server power useage** 가 있다. **EWT**가 장애변수라는 것은 우리가 쿨링타워까지는 제어하지 않는다는 의미와 동치이다. 

`-` 저자들의 목적은 다음과 같다. (1) 상태변수들을 모델링해서 다음시점에서 각 상태변수들이 어떠한 값을 가질지 예측하고 (2) 각 상태변수들이 특정한 값(set point)을 가지도록 **동시에** 제어하며 (3) 제어의 jointly **optimize** , 즉 **매우 적은 에너지만으로** 여러가지 상태변수를 원하는 값으로 동시에 제어하는 방법을 개발하는 것이다. 


`-` 고정된 시간 $t$에 대한 모델은 아래와 같다. 

$${\bf x}[t]=\sum_{k=1}^{T}A_k{\bf x}[t-k]+\sum_{k=1}^{T}B_k {\bf u}[t-k] + C {\bf d}[t-1]$$

여기에서 ${\bf x}[t]$, ${\bf u}[t]$ 그리고 ${\bf d}[t]$ 는 각각 **상태변수**, **제어변수**, **장애변수** 을 의미하는 vector이다. 또한 $A_k$, $B_k$, $C$는 각각 차원에 맞는 매트릭스이다. 

`-` 고정된 시간 $t$에 대하여 ${\bf x}[t]$가 벡터인 이유는 **상태변수** 에 해당하는 변수값이 여러개이기 때문이다. 논문에서는 DP(=differential air pressure), CAT(cold-aisle temerature), EAT(entering air temperature), LAT(leaving air temperature) 를 상태변수로 두었다. 따라서 고정된 $t$에 대하여 ${\bf x}[t]$는 4차원 벡터이다. 마찬가지 논리로 **제어변수**는 fan speed, valve opening 2개가 있으므로 ${\bf u}[t]$는 2차원 벡터이다. 장애변수는 EWT와 server power useage가 있으므로 ${\bf d}[t]$ 역시 2차원 벡터이다. 

`-` 그리고  ${\bf x}[t]$, ${\bf u}[t]$, ${\bf d}[t]$ 의 차원에 맞는 $A_k,~B_k,~C$를 각각 구해보면 $A_k=4\times 4$, $B_k$와 $C$는 $4 \times 2$ 행렬임을 알 수 있다. 

`-` 복잡해 보이는 아래식은 결국 회귀분석의 하나의 응용형태에 불과하다. 

$${\bf x}[t]=\sum_{k=1}^{T}A_k{\bf x}[t-k]+\sum_{k=1}^{T}B_k {\bf u}[t-k] + C {\bf d}[t-1]$$

총 $n$번의 시간동안 관측치를 모으게 된다면 위의 모델은 아래로 간략화하여 정리할 수 있다. 

$${\bf y}={\bf X}{\bf \beta}+\epsilon$$

여기에서 ${\bf y}$는 $n\times 4$ 인 행렬이고 ${\bf X}$는 $n \times (6T+2)$ 인 행렬이며 $\bf \beta$는 $(6T+2) \times 4$ 행렬이다. 오차항 $\epsilon$은 $n \times 4$행렬이다. 알고 싶은것 즉 ${\bf y}$와 우리가 알고 있는것 즉 ${\bf X}$를 구체적으로 명시하면 아래와 같다. 

$
{\bf y}=\begin{bmatrix}
{\bf x}[t] \\
{\bf x}[t-1] \\
\dots \\
{\bf x}[t-n+1]
\end{bmatrix}
$

$
{\bf X}=\begin{bmatrix}
{\bf x}[t-1] & \dots & {\bf x}[t-T] & {\bf u}[t-1] & \dots & {\bf u}[t-T] & {\bf d}[t-1] \\
{\bf x}[t-2] & \dots & {\bf x}[t-T-1] & {\bf u}[t-2] & \dots & {\bf u}[t-T-1] & {\bf d}[t-2] \\
\dots \\
{\bf x}[t-n] & \dots & {\bf x}[t-T-n] & {\bf u}[t-n] & \dots & {\bf u}[t-T-n] & d[t-n] 
\end{bmatrix} 
$

`-` 우리는 이제 ${\bf X} \rightarrow {\bf y}$로의 맵핑을 학습하면 되는데 이러한 맵핑정보를 요약하고 있는 $\beta$는 구체적으로 명시하면 아래와 같다. 

$$\beta=\begin{bmatrix}
A_1 \\
\dots \\
A_T \\
B_1 \\
\dots \\
B_T \\
C \\
\end{bmatrix}$$

`-` 결론적으로 말하여 저자들이 제시한 모델은 아래로 요약할 수 있다. 

$${\bf y}={\bf X}{\bf \beta}+\epsilon$$

이제 총 $n$번의 시간동안 관측치를 모으게 된다면 위의 모델에서 $\beta$는 쉽게 구할 수 있다. 그럼 $n$개의 관측치를 어떻게 모을 것인가? ${\bf y}$를 모으는 것은 별로 문제가 되지 않는다. 매 시간마다 $(DT,CAT,EAT,LAT)$ 를 꼬박꼬박 관찰하여 기록하기만 하면 된다. 문제는 ${\bf X}$를 모으는 것인데 ${\bf X}$에서는 **상태변수** ${\bf x}=(DT,CAT,EAT,LAT)$ 와 **장애변수** ${\bf d}$ 뿐만 아니라 **제어변수** ${\bf u}=(fan,valve)$ 가 포함되어 있다. 제어변수는 우리가 통제할 수 있는 변수이므로 관측하는 동안 우리가 어떻게 액션하는지에 따라서 값이 달라지는 변수이다. 그럼 $n$번의 시간동안 제어변수들을 어떻게 조절해야 하는가? 저자들은 PID를 포함한 다양한 방법으로 상태변수들이 잘 통제되고 있는 상황에서는 자료를 분석하기 어렵다고 판단하여 $n$번의 시간동안은 **랜덤하게 제어**하여 자료를 축적하였다. 쉽게 말해서 벨브를 확 열어보기도 하고 닫아보기도 하면서 어떻게 상태변수들이 변화하는지 다양한 조건에서 관찰한 것이다. 물론 이렇게 제어하면 조금 위험할 수도 있는데 (온도가 지나치게 올라간다든가? 혹은 반대라든가) 저자들은 이러한 위험한 상황을 방지하기 위해서 너무 극단적으로 밸브를 열거나 닫는 상황은 피하도록 제안하였다. 이에 대한 자세한 설명은 4.2에 나와있다. 

`-` $n$개의 관측치를 모았다면 이제 아래의 모델에서 $\beta$를 찾을 수 있다. 

$${\bf y}={\bf X}{\bf \beta}+\epsilon$$

이말은 곧 임의의 시간 $t$에 대하여 아래식을 만족하는 $A_k$, $B_k$, $C$ 값들을 정확하게 알고 있다는 것과 동치이다. 

$${\bf x}[t]=\sum_{k=1}^{T}A_k{\bf x}[t-k]+\sum_{k=1}^{T}B_k {\bf u}[t-k] + C {\bf d}[t-1]$$

즉 모델링이 완료된 상태이다. 이제 저자들의 목표 (1) 모델링이 달성되었다. 

`-` 모델링이 완료되었다는 소리는 매우 건방질수 있는데 예컨데 **적당한 조건을 우리에게 주기만 하면 우리는 다음상황에서의 상태변수 ${\bf x}$를 완벽하게 예측할 자신이 있다** 라는 의미이다. 이 말이 사실이든 거짓이든 우선 여기서는 받아들이기로 하자. 

`-` 위의 사실을 받아들이면 이제 남은 목표는 (2) 제어 (3) 효율적인 제어 이다. 여기에서 제어를 한다는 의미는 상태변수 ${\bf x}$를 원하는 set-point ${\bf x}_ {sp}$가 되도록 한다는 의미이다. 효율적인 제어라는 의미는 (2)를 달성하는데 너무 많은 에너지를 쓰지 않겠다는 의미이다. 이 방법은 너무 극단적인 액션 ${\bf u}$를 방지함으로써 얻을 수 있다. 즉 벨브를 갑자기 확 연다든가 하지 말라는 것이다. 목표 (2)와 (3)은 **적당한 제약 조건을 만족하는** 아래식을 최소화하는 ${\bf u}$를 찾음으로써 동시에 달성할 수 있다.

$$\min_ {\bf u} \sum_{\tau =t}^{t+L} \sum_{i=1}^{M} 
\left( 
\sum_{s}q_s(x_i^{s}[\tau]-x_{sp}^{s})^2+\sum_{c}r_c(u_i^{c}[\tau]-u_{min}^{c})^2
\right)$$

여기에서 $q_s$와 $r_c$는 각 상태변수와 제어변수에 해당하는 가중치이다. 그리고 $s \in \{DP,CAT,LAT\}$이며, $c \in \{fan,valve\}$이다. 



`-` 쉽게 눈치채겠지만 

$$\sum_{s}q_s(x_i^{s}[\tau]-x_{sp}^{s})^2$$

는 제어를 잘하기 위한 손실함수이다. 즉 (2)의 목표를 달성하기 위한 손실함수이다. 그리고 

$$\sum_{c}r_c(u_i^{c}[\tau]-u_{min}^{c})^2$$

는 효율적인 제어를 하기 위한 손실함수이다. 즉 (3)의 목표를 달성하기 위한 손실함수이다. 만약 $\sum_{s}q_s(x_i^{s}[\tau]-x_{sp}^{s})^2$ 항목만 있다면 제어를 잘하기 위해서 어떠한 액션 ${\bf u}$라도 불사할 것이다. 그것이 비록 에너지를 많이 소모하는 행위라 할지라도 말이다. 만약 $\sum_{c}r_c(u_i^{c}[\tau]-u_{min}^{c})^2$의 항목이 없다면 아무것도 안할 것이다. 그래야 에너지를 젤 적게 쓸테니깐 말이다. 따라서 위의 손실함수는 set-point로 도달할때 주는 **보상**과 너무 많은 에너지를 썼을때의 **벌점**을 동시에 내포하고 있다. 

`-` $q_s$는 상태변수에 대한 가중치이다. $q_s$ 설정하겠다는 것은 $(DP,CAT,LAT)$를 똑같이 중요하게 생각하지는 않겠다는 의미이다. 예를 들어 $LAT$에 해당하는 가중치가 매우 낮다면 제어를 하다가 $LAT$의 경우 set-point에 도달하지 못해도 크게 신경쓰지 않겠다는 의미이다. $r_c$역시 마찬가지로 해석할 수 있다. 

`-` 위의 식을 최소화하는 제약조건은 아래와 같다. 

(1) $u_i^c \in [u_{min}^c,u_{max}^c]$

(2) $\|u_i^c[\tau]-u_i^c[\tau-1]\| \leq \Delta^c, ~t\leq \tau \leq t+L$

(3) ${\bf d}[\tau]={\bf d}[\tau-1], ~t\leq \tau \leq t+L$

(4) ${\bf x}[t]=\sum_{k=1}^{T}A_k{\bf x}[t-k]+\sum_{k=1}^{T}B_k {\bf u}[t-k] + C {\bf d}[t-1]$

`-` 첫번째 제약조건은 너무 당연한데 밸브를 어느정도 열지 팬을 어느정도 강도로 돌릴지 결정하는 변수에는 상한값과 하한값이 있다는 것이다. 이건 안전상의 이유일수도있고 기계의 물리적인 한계때문일수도있다. 두번째 제약조건은 너무 급격한 변화가 일어나게끔 제어를 하지 말라는 것이다. 여기에서 $\Delta^c$는 각 timestep에서 허용될 수 있는 최대변화량의 절대값을 의미하는데 (논문에서는 maximum allowed absolute change 라고 적혀있다) 이건 에너지절감차원에서 이런 제약을 거는 것인지 아니면 기계의 손상을 방지하고자 이렇게 하는 것인지 잘 모르겠다. 아무튼 직관적으로 생각해도 $t-1$시점에서 벨브를 확 열었다가 $t$시점에서 벨브를 확 닫고 다시 $t+1$시점에서 벨브를 확 연다면 안 좋을것 같은 생각이 들기에 이러한 제약은 타당하여 보인다. 세번째 제약조건이 의미하는 바는 $\tau \in [t,t+L]$동안 즉 우리가 **제어모드**에 있는 동안 방해변수는 일정하다는 의미이다. 따라서 이 논문에서는 사실상 방해변수는 고려하지 않은것과 같다. 마지막 제약조건은 **적당한 값을 알면 우리(=구글)는 다음 상태변수를 정확하게 예측할 수 있다**라는 의미이다. 상당히 무리한 제약조건이지만 이건 그렇다고 믿기로 하였으므로 그냥 넘어가기로 하자. 

`-` 이후의 내용은 별것 없어보인다. 위의 최적화 문제를 풀기위해서 텐서플로를 이용했다는 것, 그리고 optimization을 위해서 adam을 사용했다는것, 방해요인이 바뀌기 전까지는 한번만 최적화하였고 그 이후에는 적절하게 초기최적상태를 업데이트 하였다는 점들이 있다. (이 부분은 논문의 수식번호 7번에 설명되어 있는데 잘 이해못하겠음, 갑자기 자기들이 잘 쓰는 방식이라면서 맥락없이 $z_i^{c}[\tau]$가 도입된다.) 그 이후에는 자기들의 컨트롤 방식이 PID에 비하여 효율적이라는 실험결과를 제시한다. 



### 소감 

`-` 아래와 같은 모델은 사실 특별한 것이 없는 모델이다. 그냥 선형모델일 뿐이다. 

$${\bf x}[t]=\sum_{k=1}^{T}A_k{\bf x}[t-k]+\sum_{k=1}^{T}B_k {\bf u}[t-k] + C {\bf d}[t-1]$$

`-` 위의 모델로 모든 값들이 모델링된다는 보장이 없다. 논문에서는 자꾸 PID보다 컨트롤 잘한다고 주장하는데 이 주장이 반드시 

$${\bf x}[t]=\sum_{k=1}^{T}A_k{\bf x}[t-k]+\sum_{k=1}^{T}B_k {\bf u}[t-k] + C {\bf d}[t-1]$$

에 의한 모델링이 정확하다고 볼 수는 없다. 위의식은 제약조건중 하나로 들어가 있을 뿐이라서 위의 제약조건상에 존재하지 않는 공간에서 ${\bf u}$의 최적화가 이루어졌다면 위의 제약조건은 아무 쓸모가 없게 되고 따라서 맞는지 틀린지도 전혀 검증할 수 없다. 실제로 논문에서 위의 모델링을 적용한 결과 모델의 MSE가 얼마라는 식의 주장(=argument)은 전혀 없다. 

`-` 논문은 ${\bf d}[\tau]={\bf d}[\tau-1]$이 성립하는 일정시간을 잘라서 분석한다. 당연히 실제 자료에서 

$${\bf d}[\tau]={\bf d}[\tau-1]$$

가 성립할리는 만무하고 적당히 

$${\bf d}[\tau] \approx {\bf d}[\tau-1]$$

느낌으로 시간을 잘라서 분석할 것이다. 그럼 어느정도 $\approx$이면 인정할 것인가? 만약에 ${\bf d}$가 서버의트래픽 때문에 생기는 발열이라면 차라리 괜찮다. 어느순간 재수없이 서버의 트래픽이 증가하여 발열이 급격하게 늘어난다면 발열이 생기기 전의 시간과 생긴후의 시간을 나누어서 분석하면 된다. 그런데 ${\bf d}$가 $EWT$와 같은 변수를 의미할때는 어떠할까? $EWT$는 물의 온도인데 시간에 따라서 일정하다가 갑자기 급격하게 변화하는 경우가 있을까? 조금씩 변화하는 함수이지 않을까? 이러한 함수에 대하여 ${\bf d}[\tau] \approx {\bf d}[\tau-1]$는 너무 순진한 가정이라고 생각한다. 